In [1]:
# import json
from pathlib import Path
import numpy as np
# import pytest
# import advanced_link_skdsp_v3_tx_flexible as txmod
import generate_tx_iq_dataset as genmod
import load_tx_iq_data as loadmod
from tx_controller_tone_pulse_stft_varlen import TonePulseTXControlNetVarLen, build_controlled_tone_pulse_from_variable_inputs
import torch
import math
import score_iq_decode as scorer
import argparse
import json
import tempfile
from pathlib import Path
from typing import Optional
import advanced_link_skdsp_v5_robust_numpy as link_numpy
import advanced_link_skdsp_v5_robust as link5
import advanced_link_skdsp_v4_robust as link4
import os
import torch.nn as nn
import torchaudio.functional as AF
import advanced_link_skdsp_v5_robust_torch as link_torch

train_path_dat = Path("C:/Users/theon/Jamming/train_set_00/dataset")

class JammerLossFunction(torch.autograd.Function):
    @staticmethod
    def forward(jammed, whole, device):
        # ctx.whole = whole
        # ctx.save_for_backward(jammed.detach())

        score = JammerLossFunction._evaluate(jammed, whole, device)  ## .cpu().numpy()
        return jammed.new_tensor(score, device = device, dtype = torch.float)

    # @staticmethod
    # def backward(ctx, grad_output):
    #     jammed, = ctx.saved_tensors
    #     whole = ctx.whole

    #     eps = 1e-4
    #     grad_jammed = torch.zeros_like(jammed)

    #     flat = jammed.reshape(-1)
    #     flat_grad = grad_jammed.reshape(-1)

    #     for i in range(flat.numel()):
    #         x_plus = jammed.clone()
    #         x_minus = jammed.clone()

    #         x_plus.reshape(-1)[i] += eps
    #         x_minus.reshape(-1)[i] -= eps

    #         f_plus = JammerLossFunction._evaluate(x_plus, whole)
    #         f_minus = JammerLossFunction._evaluate(x_minus, whole)

    #         flat_grad[i] = (f_plus - f_minus) / (2 * eps)

    #     grad_jammed = grad_jammed * grad_output
    #     return grad_jammed, None

    @staticmethod
    def _evaluate(jammed, whole, device):
        score = torch.tensor([0.0]).to(device).requires_grad_()
        try:
            rx_result = link4.rx_command_iq(jammed, whole["meta"])
            score = scorer.score_decode(rx_result, whole["meta"]).to(device).requires_grad_()
        except RuntimeError as e:
            estr = str(e)
            if estr == "No valid packet found after acquisition, header decode, FEC decode, and CRC":
                pass
            else:
                raise
        return score


class JammerLoss(nn.Module):
    def __init__(self, device = 'cuda'):
        super().__init__()
        self.device = device

    # def forward(self, jammed, whole, device):
    #     # ctx.whole = whole
    #     # ctx.save_for_backward(jammed.detach())

    #     score = self._evaluate(jammed, whole, device)  ## .cpu().numpy()
    #     return score

    def forward(self, jammed, whole):
        score = torch.tensor([0.0]).to(self.device).requires_grad_()
       
        # print('rx_result')
        rx_result = link4.rx_command_iq(jammed, whole["meta"])
        # print('score')
        if not isinstance(rx_result['message'], type(None)):
            score = torch.tensor([scorer.score_decode(rx_result, whole["meta"])]).to(self.device).requires_grad_()
            score = score + rx_result['metric_div'].requires_grad_()
        return score

    # def forward(self, jammed, whole):
    #     return JammerLossFunction.apply(jammed, whole, self.device)

def repeat_to_length_mod(arr, target_length):
    if arr.ndim != 1:
        raise ValueError("Input tensor must be 1D")
    if arr.numel() == 0:
        raise ValueError("Input tensor must not be empty")

    idx = torch.arange(target_length, device=arr.device) % arr.numel()
    return arr[idx]


def delete_if_exists(filepath):
    """
    Check if a file exists and delete it.

    Parameters:
        filepath (str): Path to the file
    """
    if os.path.isfile(filepath):
        os.remove(filepath)
        print(f"Deleted: {filepath}")
    else:
        print(f"File not found: {filepath}")

# tmp_path = Path("local_test")
# out_root = tmp_path / "dataset"

# produced = genmod.generate_dataset(
#                                     output_root=out_root,
#                                     num_outputs=4,
#                                     min_total_samples=8192,
#                                     max_total_samples=12000,
#                                     section_len=1024,
#                                     num_sections=3,
#                                     seed=3,
#                                     random_payload_probability=0.5,
#                                     )

# assert len(produced) == 4

# sample_dirs = loadmod.list_sample_dirs(out_root)
# assert len(sample_dirs) == 4

# for sdir in sample_dirs:
#     assert (sdir / "whole_iq.npy").exists()
#     assert (sdir / "whole_meta.json").exists()
#     assert (sdir / "sections.npy").exists()
#     assert (sdir / "sections_meta.json").exists()

# whole = loadmod.load_whole_iq(sdir)
# sections = loadmod.load_sections(sdir)
# bundle = loadmod.load_sample_bundle(sdir)

# iq = whole["iq"]
# whole_meta = whole["meta"]
# secs = sections["sections"]
# sections_meta = sections["meta"]

# assert iq.dtype == np.complex64
# assert secs.dtype == np.complex64
# assert secs.shape == (3, 1024)

# assert whole_meta["actual_num_samples"] == len(iq)
# assert sections_meta["whole_num_samples"] == len(iq)
# assert len(sections_meta["starts"]) == 3

# for idx, start in enumerate(sections_meta["starts"]):
#     np.testing.assert_array_equal(secs[idx], iq[start:start + 1024])

# assert bundle["whole_iq"].shape == iq.shape
# assert bundle["sections"].shape == secs.shape

print('ready')

ready


In [2]:
device = "cuda"

epochs = 10

jammer_sampling_freq = 2e9
scoring_dict = {}

model = TonePulseTXControlNetVarLen(in_ch=4, base_ch=16, n_scalar_features=6, max_tones=8).to(device)
returns = []
criterion = JammerLoss(device)

optimizer = torch.optim.Adam(model.parameters(), lr=5e-2)

model.train()
loop_limit=3

for epoch in range(epochs):
    # epoch_score = 0.0
    print(f'epoch : {epoch}')
    sample_dirs = loadmod.list_sample_dirs(train_path_dat)
    scores=[]
    loop_limiter = 0 
    for sdir in sample_dirs:
        # if loop_limiter == loop_limit:
        #     break
        whole = loadmod.load_whole_iq(sdir)
        # loop_limiter+=1
        whole_iq = whole['iq']
        sections = loadmod.load_sections(sdir)
        iq1 = sections['sections'][0]
        iq2 = sections['sections'][1]
        iq3 = sections['sections'][2]
        whole_sample_rate = whole['meta']['sample_rate_hz']
        
        # rx_result_check = link.rx_command_iq(whole_iq, whole["meta"])
        # score_check = scorer.score_decode(rx_result_check, whole["meta"])
        
        # print('resampling 1')
        iq1 = link4.resample_iq(iq1, whole_sample_rate, jammer_sampling_freq)[:1024]
        # print('resampling 2')
        iq2 = link4.resample_iq(iq2, whole_sample_rate, jammer_sampling_freq)[:1024]
        # print('resampling 3')
        iq3 = link4.resample_iq(iq3, whole_sample_rate, jammer_sampling_freq)[:1024]
        
        # print('jam_iq')
        jam_iq =  build_controlled_tone_pulse_from_variable_inputs(
                                                                    model=model,
                                                                    rx_iq_windows=[iq1, iq2, iq3],
                                                                    intake_sample_rate_hz=jammer_sampling_freq,
                                                                    desired_output_iq_len=8_000,
                                                                    user_peak_power_fraction=40.0,
                                                                    seed=11,
                                                                    device=device,
                                                                )

        # print('jam resampling')

        jam_iq_rx_resam = link4.resample_iq(jam_iq['tx_iq'], jammer_sampling_freq, whole_sample_rate).requires_grad_()
        jam_iq_rx_resam = repeat_to_length_mod(jam_iq_rx_resam, whole_iq.shape[0]).requires_grad_()
        jammed = torch.tensor(whole_iq, requires_grad=True).to(device) + jam_iq_rx_resam[:whole_iq.shape[0]].requires_grad_()
        # jammed = torch.tensor(jammed, device ='cuda', dtype=torch.cfloat).requires_grad_(True)
        # print('scoring')
        scores.append(criterion(jammed, whole))
       
        # epoch_score += scores[-1].item()
        print(f'epoch : {epoch}  sdir : {sdir} score: {scores[-1].item()}')

    loss = torch.stack(scores).sum()/len(scores)

    print(loss.requires_grad)  # True
    print(loss.grad_fn)        # something like <MseLossBackward>

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()


    scoring_dict[epoch] = loss
    print(scoring_dict)

scoring_dict

epoch : 0
epoch : 0  sdir : C:\Users\theon\Jamming\train_set_00\dataset\sample_000000 score: 0.0
epoch : 0  sdir : C:\Users\theon\Jamming\train_set_00\dataset\sample_000001 score: 10.772265434265137
epoch : 0  sdir : C:\Users\theon\Jamming\train_set_00\dataset\sample_000002 score: 10.760795593261719
epoch : 0  sdir : C:\Users\theon\Jamming\train_set_00\dataset\sample_000003 score: 1.9038195610046387
epoch : 0  sdir : C:\Users\theon\Jamming\train_set_00\dataset\sample_000004 score: 0.0
epoch : 0  sdir : C:\Users\theon\Jamming\train_set_00\dataset\sample_000005 score: 10.808090209960938
epoch : 0  sdir : C:\Users\theon\Jamming\train_set_00\dataset\sample_000006 score: 10.770838737487793
epoch : 0  sdir : C:\Users\theon\Jamming\train_set_00\dataset\sample_000007 score: 0.0
epoch : 0  sdir : C:\Users\theon\Jamming\train_set_00\dataset\sample_000008 score: 0.0
epoch : 0  sdir : C:\Users\theon\Jamming\train_set_00\dataset\sample_000009 score: 10.830608367919922
epoch : 0  sdir : C:\Users\the

{0: tensor(4.9099, device='cuda:0', grad_fn=<DivBackward0>),
 1: tensor(5.0764, device='cuda:0', grad_fn=<DivBackward0>),
 2: tensor(4.9959, device='cuda:0', grad_fn=<DivBackward0>),
 3: tensor(5.1658, device='cuda:0', grad_fn=<DivBackward0>),
 4: tensor(4.9194, device='cuda:0', grad_fn=<DivBackward0>),
 5: tensor(5.0823, device='cuda:0', grad_fn=<DivBackward0>),
 6: tensor(4.9912, device='cuda:0', grad_fn=<DivBackward0>),
 7: tensor(5.2223, device='cuda:0', grad_fn=<DivBackward0>),
 8: tensor(4.9865, device='cuda:0', grad_fn=<DivBackward0>),
 9: tensor(4.9939, device='cuda:0', grad_fn=<DivBackward0>)}

In [5]:
scoring_dict

{0: tensor(5.0010, device='cuda:0', grad_fn=<DivBackward0>),
 1: tensor(4.8305, device='cuda:0', grad_fn=<DivBackward0>),
 2: tensor(4.9094, device='cuda:0', grad_fn=<DivBackward0>)}

In [8]:
train_path = Path("C:/Users/theon/Jamming/train_set_00")
# test_path = Path("test_set_00")
# val_path = Path("val_set_00")

# for _ in [train_path, test_path, val_path]:
out_root = train_path / "dataset"
produced = genmod.generate_dataset(
                                    output_root=out_root,
                                    num_outputs=500,
                                    min_total_samples=4000,
                                    max_total_samples=5500,
                                    section_len=1024,
                                    num_sections=3,
                                    seed=3,
                                    random_payload_probability=0.0,
                                    )

out_root = test_path / "dataset"
produced = genmod.generate_dataset(
                                    output_root=out_root,
                                    num_outputs=500,
                                    min_total_samples=4000,
                                    max_total_samples=5500,
                                    section_len=1024,
                                    num_sections=3,
                                    seed=3,
                                    random_payload_probability=0.0,
                                    )

sucessfully generated data
sucessfully generated data
sucessfully generated data
sucessfully generated data
sucessfully generated data
sucessfully generated data
sucessfully generated data
sucessfully generated data
sucessfully generated data
sucessfully generated data
sucessfully generated data
sucessfully generated data
sucessfully generated data
sucessfully generated data
sucessfully generated data
sucessfully generated data
sucessfully generated data
sucessfully generated data
sucessfully generated data
sucessfully generated data
sucessfully generated data
sucessfully generated data
sucessfully generated data
sucessfully generated data
sucessfully generated data


In [2]:
train_path = Path("C:/Users/theon/Jamming/test_set_00")
# test_path = Path("test_set_00")
# val_path = Path("val_set_00")

# for _ in [train_path, test_path, val_path]:
out_root = train_path / "dataset"
produced = genmod.generate_dataset(
                                    output_root=out_root,
                                    num_outputs=25,
                                    min_total_samples=5000,
                                    max_total_samples=8000,
                                    section_len=1024,
                                    num_sections=3,
                                    seed=3,
                                    random_payload_probability=0.0,
                                    )

sucessfully generated data
sucessfully generated data
sucessfully generated data
sucessfully generated data
sucessfully generated data
sucessfully generated data
sucessfully generated data
sucessfully generated data
sucessfully generated data
sucessfully generated data
sucessfully generated data
sucessfully generated data
sucessfully generated data
sucessfully generated data
sucessfully generated data
sucessfully generated data
sucessfully generated data
sucessfully generated data
sucessfully generated data
sucessfully generated data
sucessfully generated data
sucessfully generated data
sucessfully generated data
sucessfully generated data
sucessfully generated data


In [13]:
# "C:\Users\theon\Jamming\train_set_00"
# train_path_dat = Path("C:/Users/theon/Jamming/test_set_00/dataset")
sample_dirs = loadmod.list_sample_dirs(train_path_dat)
sdir = sample_dirs[0]
whole = loadmod.load_whole_iq(sdir)
# for sdir in sample_dirs:
#     assert (sdir / "whole_iq.npy").exists()
#     assert (sdir / "whole_meta.json").exists()
#     assert (sdir / "sections.npy").exists()
#     assert (sdir / "sections_meta.json").exists()

# whole = loadmod.load_whole_iq(sdir)
# sections = loadmod.load_sections(sdir)
sdir

WindowsPath('C:/Users/theon/Jamming/train_set_00/dataset/sample_000000')

In [14]:
whole["meta"]

{'payload_source': 'message',
 'message_length': 776,
 'message': '8YtDt XbiufMdI8X2Y4rUmer BH3M1 XS0DRdJuPnBIKpi99lSi0tcL21pffWQJ EeNajnez0LHtfROUrWW6 Xn I3EM3H MRb1OcWrhQ7TTJ chcVG6MOwUxOVHMWndqN CIEPx3mnPQC4vkRB5ICpeyOxJRkSq1L I7S1L10e0tza9 3Ce6 KRDiKpFfe z3gb9pv MEc0goRqG9hTCzppvEJqa ZgIFI 2g8Pahqfpgi9el   OuOjSXXMUHyQ2pqaWkwfV6Wf3gV O116cFBIj2C2qdPVHp7pWnOna8sEXflmWwdRpdo9KMle EnmhPxjExF6YGVYS1kW E0u1 9tZtum 9 4xrIzs ODL0IBNcI9WzwUEP9s19A7d9jZf7DEiBGEyHrxeGvfOx2lkplHLeT5RadQQ3W jA YqOpJj3o4GmVV5LHnRo  ThLxtwV6pnsQ1Mx69NwinxYTmIIXgrf9 IF TQZ5iT otImooxy1YqsYyvwzGVLd40XON M9dyanD w6zyBe 4oKtr7lgdUD j cRPQSrkekRAiz3C Onf0jzuY 8i2A Mc76Z4x6eGUV5UZCaAHVs6yuAcvZ vdrov4 xhcZ5O0egEZfY dCEmX8yvQoSpgLJ7M FIdRSOlh3landlcv e9gy Qz9R9SeWNYlLx0o XQZwXTxU14D49SIv X ftvc7lmOEhg57QVajyZnRNo5kAEgtsboDKAC 1 Oy7wkfodmzGkn7ZCn SZ5oL4WAoa7MjRSy jU2',
 'random_bits': None,
 'random_seed': None,
 'payload_len': 776,
 'payload_len_bytes': 776,
 'target_num_samples': None,
 'actual_num_samples': 123040,
 '

In [15]:
whole = loadmod.load_whole_iq(sdir)
        
whole_iq = whole['iq']

genmod._decode_candidate_with_v4(whole_iq, whole["meta"])

{'payload_bytes': b'8YtDt XbiufMdI8X2Y4rUmer BH3M1 XS0DRdJuPnBIKpi99lSi0tcL21pffWQJ EeNajnez0LHtfROUrWW6 Xn I3EM3H MRb1OcWrhQ7TTJ chcVG6MOwUxOVHMWndqN CIEPx3mnPQC4vkRB5ICpeyOxJRkSq1L I7S1L10e0tza9 3Ce6 KRDiKpFfe z3gb9pv MEc0goRqG9hTCzppvEJqa ZgIFI 2g8Pahqfpgi9el   OuOjSXXMUHyQ2pqaWkwfV6Wf3gV O116cFBIj2C2qdPVHp7pWnOna8sEXflmWwdRpdo9KMle EnmhPxjExF6YGVYS1kW E0u1 9tZtum 9 4xrIzs ODL0IBNcI9WzwUEP9s19A7d9jZf7DEiBGEyHrxeGvfOx2lkplHLeT5RadQQ3W jA YqOpJj3o4GmVV5LHnRo  ThLxtwV6pnsQ1Mx69NwinxYTmIIXgrf9 IF TQZ5iT otImooxy1YqsYyvwzGVLd40XON M9dyanD w6zyBe 4oKtr7lgdUD j cRPQSrkekRAiz3C Onf0jzuY 8i2A Mc76Z4x6eGUV5UZCaAHVs6yuAcvZ vdrov4 xhcZ5O0egEZfY dCEmX8yvQoSpgLJ7M FIdRSOlh3landlcv e9gy Qz9R9SeWNYlLx0o XQZwXTxU14D49SIv X ftvc7lmOEhg57QVajyZnRNo5kAEgtsboDKAC 1 Oy7wkfodmzGkn7ZCn SZ5oL4WAoa7MjRSy jU2',
 'payload_len': 776,
 'message': '8YtDt XbiufMdI8X2Y4rUmer BH3M1 XS0DRdJuPnBIKpi99lSi0tcL21pffWQJ EeNajnez0LHtfROUrWW6 Xn I3EM3H MRb1OcWrhQ7TTJ chcVG6MOwUxOVHMWndqN CIEPx3mnPQC4vkRB5ICpeyOxJRkSq1L I7S1

In [16]:
whole = loadmod.load_whole_iq(sdir)
        
whole_iq = whole['iq']
rx_result_check = link5.rx_command_iq(whole_iq, whole["meta"])
score_check = scorer.score_decode(rx_result_check, whole["meta"])
score_check

coarse_start: 0, coarse_cfo_hz : 14500.0, coarse_metric : 1162.5411466955536


1.0

In [4]:
whole = loadmod.load_whole_iq(sdir)
        
whole_iq = whole['iq']
rx_result_check = link4.rx_command_iq(whole_iq, whole["meta"])
score_check = scorer.score_decode(rx_result_check, whole["meta"])
score_check

from loop best_start : 0, type : <class 'int'>
from loop best_cfo : 0.0, type : <class 'float'>
from loop best_metric : 1157.6636962890625, type : <class 'float'>
coarse_start: 0, coarse_cfo_hz : 0.0, coarse_metric : 1157.6636962890625


1.0

In [6]:
whole = loadmod.load_whole_iq(sdir)
        
whole_iq = whole['iq']
rx_result_check = link_numpy.rx_command_iq(whole_iq, whole["meta"])
score_check = scorer.score_decode(rx_result_check, whole["meta"])
score_check

1.0

In [9]:
whole = loadmod.load_whole_iq(sdir)
        
whole_iq = whole['iq']
rx_result_check = link_torch.rx_command_iq(whole_iq, whole["meta"])
score_check = scorer.score_decode(rx_result_check, whole["meta"])
score_check

RuntimeError: Input type (CUDAComplexDoubleType) and weight type (CUDAComplexFloatType) should be the same

In [11]:
link_torch.bpsk_map([1,0])

tensor([ 1.+0.j, -1.+0.j], device='cuda:0')

In [13]:
link_numpy.bpsk_map([1,0,1,0])

array([ 1.+0.j, -1.+0.j,  1.+0.j, -1.+0.j], dtype=complex64)

In [14]:
link_torch.tx_waveform(bits = [1.0], sps = 8, beta = 0.35, span = 6)

tensor([-0.0021+0.j, -0.0022+0.j, -0.0016+0.j, -0.0005+0.j,  0.0010+0.j,  0.0023+0.j,
         0.0032+0.j,  0.0034+0.j], device='cuda:0')

In [15]:
link_numpy.tx_waveform(bits = [1.0], sps = 8, beta = 0.35, span = 6)

array([-0.00207092+0.j, -0.0021541 +0.j, -0.00158815+0.j, -0.00046041+0.j,
        0.00097236+0.j,  0.00233953+0.j,  0.00324914+0.j,  0.00339539+0.j],
      dtype=complex64)

In [6]:
whole = loadmod.load_whole_iq(sdir)
        
whole_iq = whole['iq']
rx_result_check = link_numpy.rx_command_iq(whole_iq, whole["meta"])
score_check = scorer.score_decode(rx_result_check, whole["meta"])
score_check

RuntimeError: No valid packet found after acquisition, header decode, FEC decode, and CRC

In [6]:
sdir

WindowsPath('C:/Users/theon/Jamming/train_set_00/dataset/sample_000004')

In [4]:
iq_path = Path(sdir / "whole_iq.npy")
meta_path = Path(sdir / "whole_meta.json")
print(f'sdir : {sdir}')
# assert (sdir / "whole_iq.npy").exists()
#     assert (sdir / "whole_meta.json").exists()
               
score = scorer.main(["--iq", str(iq_path), "--metadata", str(meta_path)])
score

sdir : C:\Users\theon\Jamming\train_set_00\dataset\sample_000000
from loop best_start : 1065, type : <class 'int'>
from loop best_cfo : 7500.0, type : <class 'float'>
from loop best_metric : 216.97389221191406, type : <class 'float'>
0.0


0.0

In [5]:
whole = loadmod.load_whole_iq(sdir)
        
whole_iq = whole['iq']
# whole_iq
rx_result_check = link.rx_command_iq(whole_iq, whole["meta"])
# score_check = scorer.score_decode(rx_result_check, whole["meta"])

RuntimeError: No valid packet found after acquisition, header decode, FEC decode, and CRC

In [5]:
sections = loadmod.load_sections(sdir)
sections['sections'][0].shape

(1024,)

In [6]:
whole['iq'].shape[0]

123040

In [19]:
sdir_01 = sample_dirs[1]
whole_01 = loadmod.load_whole_iq(sdir_01)
whole_01['meta']['fec']


'conv'

In [6]:
score_check = scorer.score_decode(rx_result_check, whole_01["meta"])
score_check

1.0

In [3]:
iq_path = Path(sample_dirs[4] / "whole_iq.npy")
meta_path = Path(sample_dirs[4] / "whole_meta.json")
print(f'sdir : {sample_dirs[4]}')
# assert (sdir / "whole_iq.npy").exists()
#     assert (sdir / "whole_meta.json").exists()
               
score = scorer.main(["--iq", str(iq_path), "--metadata", str(meta_path)])
score

sdir : C:\Users\theon\Jamming\train_set_00\dataset\sample_000004
from broadcast best_start : 59017, type : <class 'int'>
from broadcast best_cfo : 11000.0, type : <class 'float'>
from broadcast best_metric : 239.02914428710938, type : <class 'float'>
0.0


0.0

In [22]:
score = JammerLossFunction._evaluate(whole_01['iq'], whole_01)
type(score)

float

In [39]:
jl = JammerLoss()
jt = torch.tensor(whole_01['iq'], device ='cuda', dtype=torch.cfloat)
jt.shape
    

torch.Size([114080])

In [40]:
whole_01['iq'].shape

(114080,)

In [41]:
whole_01['iq']

array([-0.01453683+0.00050056j, -0.00456747+0.00483577j,
       -0.00360805+0.00612219j, ...,  0.37030873-0.07326386j,
        0.36604854-0.06185472j,  0.00041609+0.00297634j],
      shape=(114080,), dtype=complex64)

In [43]:
jt.cpu().numpy().shape


(114080,)

In [23]:
score

1.0

In [9]:
scoring_dict

{0: 100.0}

epoch : 0


C:\Users\theon\AppData\Local\Temp\ipykernel_9948\57984809.py:60: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  jammed = torch.tensor(jammed, device ='cuda', dtype=torch.cfloat).requires_grad_(True)


RuntimeError: expected scalar type Float but found ComplexFloat

In [ ]:
def compute_returns(rewards, gamma: float):
    """
    Compute discounted returns for one episode.
    """
    returns = []
    G = 0.0
    for r in reversed(rewards):
        G = r + gamma * G
        returns.append(G)
    returns.reverse()
    return returns


def train_policy_gradient(
    env,
    policy: nn.Module,
    optimizer: optim.Optimizer,
    num_episodes: int = 1000,
    gamma: float = 0.99,
    device: str = "cpu",
    max_steps_per_episode: int = 1000,
    normalize_returns: bool = True,
    print_every: int = 50,
):
    """
    Generic REINFORCE training loop.

    Args:
        env: Gym-like environment with reset() and step(action)
        policy: PyTorch model mapping observation -> action logits
        optimizer: PyTorch optimizer
        num_episodes: number of training episodes
        gamma: discount factor
        device: torch device
        max_steps_per_episode: safety cap per episode
        normalize_returns: whether to normalize discounted returns
        print_every: logging frequency
    """
    device = torch.device(device)
    policy.to(device)

    reward_history = []

    for episode in range(1, num_episodes + 1):
        # Gymnasium reset API compatibility
        reset_result = env.reset()
        state = reset_result[0] if isinstance(reset_result, tuple) else reset_result

        log_probs = []
        rewards = []
        total_reward = 0.0

        for step in range(max_steps_per_episode):
            action, log_prob = select_action(policy, state, device)

            step_result = env.step(action)
            if len(step_result) == 5:
                next_state, reward, terminated, truncated, _ = step_result
                done = terminated or truncated
            else:
                next_state, reward, done, _ = step_result

            log_probs.append(log_prob)
            rewards.append(reward)
            total_reward += reward
            state = next_state

            if done:
                break

        # Compute discounted returns
        returns = compute_returns(rewards, gamma)
        returns = torch.tensor(returns, dtype=torch.float32, device=device)

        if normalize_returns and len(returns) > 1:
            returns = (returns - returns.mean()) / (returns.std() + 1e-8)

        # Policy gradient loss
        loss = []
        for log_prob, G in zip(log_probs, returns):
            loss.append(-log_prob * G)
        loss = torch.stack(loss).sum()

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        reward_history.append(total_reward)

        if episode % print_every == 0:
            avg_reward = sum(reward_history[-print_every:]) / print_every
            print(
                f"Episode {episode:4d} | "
                f"Avg Reward: {avg_reward:8.3f} | "
                f"Last Reward: {total_reward:8.3f} | "
                f"Loss: {loss.item():8.3f}"
            )

    return reward_history